# 23-1 Maching Learning Term project


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn
import statsmodels.formula.api as smf 

In [ ]:
# Preprocessing
data = pd.DataFrame(columns=['Count', 'IsCorrect_mean', 'Confidence'])

# IsCorrect? ???, Question? ?? ??(Count)? ??
train_task_data = pd.read_csv('data/train_data/train_task_3_4.csv')
for k,df in train_task_data.groupby('QuestionId'):
    data.loc[k] = len(df), df['IsCorrect'].mean(), 0

# Confidence? ?? ???? Confidence? ???? ??
answer_metadata = pd.read_csv('data/meta_data/answer_metadata_task_3_4.csv')

# Confidence ?? NaN? ? ??
nan_rows = answer_metadata[answer_metadata['Confidence'].isna()]
confidence_mean = answer_metadata.loc[~answer_metadata['Confidence'].isna(), 'Confidence'].mean()
answer_metadata.loc[nan_rows.index, 'Confidence'] = confidence_mean

# AnswerId? 1:1 ???? QuestionId AnswerId Confidence ???? ??? ??? DataFrame ??
merged_data = pd.merge(train_task_data, answer_metadata, on='AnswerId')[['QuestionId', 'AnswerId', 'Confidence']]
merged_data = merged_data.groupby(['QuestionId', 'AnswerId']).mean().reset_index()

In [ ]:
# data DataFrame? Confidence ? ???
# [??] Colab ??, 4? ??? ??? ????.
for i, row in merged_data.iterrows():
    data.loc[row['QuestionId'], 'Confidence'] = row['Confidence'] - np.random.randint(confidence_mean)

data

In [ ]:
# Methods

lm = smf.ols(formula='I(data.index) ~ (Confidence + (abs(50-IsCorrect_mean))/Count) - I(data.index/1000)', data=data)
lm_fit = lm.fit()
lm_fit.summary()

In [ ]:
plt.scatter(data.index, ((data.Confidence) + abs(50-data.IsCorrect_mean)/data.Count) + data.index/1000)

In [ ]:
# ??? ??
predicted = lm_fit.predict(data)

# ??? ???? ?????? ??
data_sorted = data.iloc[predicted.argsort()[::-1]]

# ??? ???????? index ??
index_sorted = data_sorted.index

# ??, ???????? y? ?? ???? ??? ??? ?????? ??
sorted_data = data.loc[index_sorted]

# ??? ???????? index? ???? QuestionId? ranking ??? ?? ???? ??
template = pd.DataFrame({'QuestionId': sorted_data.index, 'ranking': range(1, len(data)+1)})

# csv ??? ??
template.to_csv('submission/template.csv', index=False)

## 841, 660, 847, 422, 733, 928, 511, 315, 429, 153 ...
## ... 931, 282, 599, 856, 182,  93, 276, 155, 528, 500

In [ ]:
# ??

template = pd.read_csv('submission/template.csv')
len(template)

In [ ]:
template.head()

In [ ]:
eval_dev = pd.read_csv('data/test_data/quality_response_remapped_private.csv')

def evaluate(truth, submission):
    # extract ranking
    left = list(truth.left)
    right = list(truth.right)
#     if len(left) != len(right):
#     message = 'left and right lengths are not the same'
#     sys.exit(message)

    submission_left = []
    submission_right = []
    submission_preference = []
    for idx in range(len(left)):
        submission_left.append(left[idx])
        submission_right.append(right[idx])
        ranking_left = submission[submission.QuestionId==left[idx]].ranking.values[0]
        ranking_right = submission[submission.QuestionId==right[idx]].ranking.values[0]
        preference = 1 if ranking_left < ranking_right else 2
        submission_preference.append(preference)
    print(submission_preference)
    
evaluate(eval_dev, template)